# Notebook 01 — Business Understanding

> **Projet :** Détection d'anomalies et de fraudes dans les transactions financières  
> **Client :** PwC Tunisie  
> **Auteur :** Yousra Chaieb  
> **Date :** Juin 2026  

---

Ce notebook constitue la première étape de la méthodologie CRISP-DM.  
Il définit le cadre métier du projet avant toute exploration des données.

## 1.1 Contexte du projet

### Secteur et environnement

Les systèmes de paiement mobile et de transfert électronique de fonds connaissent une **croissance exponentielle** à l'échelle mondiale. En Afrique, les plateformes de mobile money ont permis l'inclusion financière de millions de personnes non bancarisées, mais ont simultanément ouvert de nouveaux vecteurs de fraude.

Les données utilisées dans ce projet ont été fournies par un **client anonyme** dans le cadre d'une mission de conseil réalisée par **PwC Tunisie**. Elles couvrent un historique complet de transactions de paiement mobile sur une période d'un mois :

| Caractéristique | Valeur |
|---|---|
| Nombre total de transactions | 6 362 620 |
| Durée couverte | 30 jours (720 heures) |
| Transactions frauduleuses | 8 213 (0,13 %) |
| Types de transactions | PAYMENT, TRANSFER, CASH_OUT, CASH_IN, DEBIT |
| Source | Client anonyme — Mission Risk Advisory PwC Tunisie |

### Cadre du projet

Ce projet s'inscrit dans le cadre d'un **stage de fin d'études** réalisé au sein du département Data & Analytics de **PwC Tunisie**. L'objectif est de produire un pipeline de détection de fraudes reproductible, explicable et opérationnel, répondant aux exigences de la mission de conseil.

## 1.2 Présentation de la problématique métier

### Le problème de la fraude financière

La fraude dans les systèmes de paiement mobile représente un **risque opérationnel et financier majeur** pour les établissements financiers. Elle se manifeste principalement par :

- **L'usurpation de compte** : un fraudeur prend le contrôle d'un compte légitime et vide le solde via des TRANSFER et CASH_OUT successifs.
- **La manipulation de solde** : des transactions conçues pour masquer l'origine frauduleuse des fonds (solde de destination anormalement nul avant réception).
- **L'exploitation des fenêtres nocturnes** : les heures creuses (0h–9h, 23h) présentent un taux de fraude jusqu'à **10,6× supérieur** à la moyenne journalière.

### Pourquoi la détection automatique est indispensable

| Contrainte | Impact sans automatisation |
|---|---|
| Volume (~200 000 tx/jour analysées) | Impossible à analyser manuellement |
| Délai de détection | La fraude se consomme en quelques minutes |
| Ratio 1:774 (fraude/normal) | Les règles heuristiques génèrent trop de faux positifs |
| Évolution des schémas de fraude | Les règles statiques deviennent obsolètes |

### Enjeux spécifiques identifiés

L'analyse préliminaire des données révèle que :
- **Seuls les TRANSFER et CASH_OUT** contiennent des fraudes — les autres types sont intrinsèquement sûrs.
- Le flag natif `isFlaggedFraud` du système existant ne détecte que **0,39 %** des fraudes réelles (Recall = 0,0039), ce qui constitue la baseline à battre.
- La variable la plus discriminante est `balance_diff_orig` (corrélation de Spearman = 0,37) : lors d'une fraude, le compte source est systématiquement vidé.

## 1.3 Objectifs métier

Les objectifs sont définis en concertation avec l'équipe Risk Advisory de PwC Tunisie et le client final.

### Objectif principal

> **Réduire les pertes financières liées à la fraude** en détectant automatiquement les transactions suspectes avec un minimum de faux positifs perturbant l'expérience client.

### Objectifs secondaires

| # | Objectif | Priorité |
|---|---|---|
| O1 | Détecter les fraudes en temps quasi-réel (< 1 seconde par transaction) | Haute |
| O2 | Minimiser les pertes financières directes (maximiser le Recall) | Haute |
| O3 | Contrôler les faux positifs pour ne pas bloquer des clients légitimes | Haute |
| O4 | Fournir des explications compréhensibles par les analystes fraude | Moyenne |
| O5 | Générer des rapports automatiques en langage naturel pour les responsables | Moyenne |
| O6 | Fonctionner sans étiquettes de fraude (détection non supervisée) | Basse |
| O7 | Fournir un pipeline reproductible et maintenable | Basse |

### Parties prenantes

| Partie prenante | Rôle | Attente principale |
|---|---|---|
| Équipe Risk Advisory PwC | Maîtrise d'ouvrage | Pipeline livrable et documenté |
| Analystes fraude (client) | Utilisateurs finaux | Alertes actionnables avec explications |
| Direction financière (client) | Décideur | Réduction des pertes, ROI mesurable |
| Équipe IT (client) | Intégrateur | API/modèle déployable en production |

## 1.4 Traduction du besoin métier en problème Data Science

### Reformulation formelle

Le besoin métier se traduit par **deux sous-problèmes complémentaires** :

#### Sous-problème 1 : Classification supervisée binaire

$$y \in \{0 \text{ (normal)}, 1 \text{ (fraude)}\}$$

Étant donné un vecteur de features $\mathbf{x} \in \mathbb{R}^{14}$ extrait d'une transaction, apprendre une fonction $f : \mathbf{x} \mapsto \hat{y}$ qui maximise la détection des fraudes.

**Défi principal :** déséquilibre extrême des classes — ratio 1:774 (0,13 % de fraudes).

#### Sous-problème 2 : Détection d'anomalies non supervisée

Sans utiliser les étiquettes de fraude, apprendre la distribution des transactions normales et détecter comme anomalies les transactions dont la reconstruction dépasse un seuil $\theta^*$ :

$$\text{score}(\mathbf{x}) = \|\mathbf{x} - \hat{\mathbf{x}}\|^2 > \theta^* \Rightarrow \text{fraude}$$

### Pipeline Data Science retenu

```
Données brutes (6.36M tx)
        │
        ▼
  [NB02] Compréhension des données (EDA)
        │  → Distribution des variables, corrélations, patterns temporels
        ▼
  [NB03] Préparation des données
        │  → Feature engineering (14 features), suppression des fuites,
        │    log-transform, split 70/15/15, SMOTE, StandardScaler
        ▼
  [NB04] Modèles de base (supervisés)
        │  → Logistic Regression, Random Forest, XGBoost, Isolation Forest
        ▼
  [NB05] AutoEncoder (non supervisé)
        │  → Entraîné sur transactions normales uniquement
        │    Score = erreur de reconstruction
        ▼
  [NB06] Explicabilité (SHAP + LIME)
        │  → Importance globale des features, explications locales
        ▼
  [NB07] Intégration LLM
           → Génération de rapports en langage naturel (Groq API)
```

### Features engineered (14 au total)

| Catégorie | Features | Justification métier |
|---|---|---|
| **Temporelles** | `hour`, `day`, `is_night`, `is_weekend` | Fraudes concentrées la nuit et en semaine |
| **Comportementales** | `balance_diff_orig`, `balance_diff_dest`, `balance_ratio_orig` | Vidage systématique du compte source |
| **Transaction** | `log_amount`, `type_TRANSFER`, `type_CASH_OUT`, `type_PAYMENT`, `type_CASH_IN`, `type_DEBIT` | Seuls TRANSFER/CASH_OUT sont frauduleux |

## 1.5 Critères de succès métier

### Définition de la réussite du point de vue client

| Critère | Description | Seuil cible |
|---|---|---|
| **Taux de détection** | Proportion de fraudes réelles correctement identifiées | ≥ 80 % |
| **Taux de faux positifs** | Proportion de transactions légitimes bloquées à tort | ≤ 1 % |
| **Latence de scoring** | Temps de réponse du modèle par transaction | < 1 seconde |
| **Lisibilité des alertes** | Les analystes fraude comprennent pourquoi une alerte est levée | Validation utilisateur |
| **Valeur du rapport LLM** | Les rapports automatiques sont jugés utiles par les responsables | Score ≥ 3,5/5 |

### Calcul de l'impact financier attendu

En supposant que le montant moyen d'une transaction frauduleuse est de **~179 500 unités monétaires** (valeur médiane observée dans les données client) :

- Sur 258 fraudes (échantillon de 200k tx) : si l'on détecte 80 % → 206 fraudes bloquées
- Réduction des pertes estimée : **206 × 179 500 ≈ 37M d'unités**
- Le baseline actuel (`isFlaggedFraud`) ne bloque qu'**1 fraude sur 258** dans l'échantillon

> Le passage du système de règles heuristiques existant à un modèle ML représente une amélioration de **200× du Recall** sur l'objectif.

## 1.6 Critères de succès Data Science

### Métriques d'évaluation retenues

Compte tenu du **fort déséquilibre des classes** (0,13 % de fraudes), l'accuracy est une métrique non pertinente (un modèle prédisant toujours "normal" atteint 99,87 % d'accuracy). Les métriques retenues sont :

| Métrique | Formule | Justification |
|---|---|---|
| **Recall** | TP / (TP + FN) | Priorité : ne pas manquer de fraude → minimiser FN |
| **Precision** | TP / (TP + FP) | Contrôler les faux positifs pour l'expérience client |
| **F2-score** | (1+4)·P·R / (4P+R) | Pondère le Recall 2× plus que la Precision |
| **PR-AUC** | Aire sous la courbe Precision-Recall | Robuste au déséquilibre, meilleure que ROC-AUC |
| **ROC-AUC** | Aire sous la courbe ROC | Comparaison standard entre modèles |

### Seuils de performance cibles par modèle

| Modèle | Recall cible | PR-AUC cible | Justification |
|---|---|---|---|
| Logistic Regression | ≥ 0,65 | ≥ 0,60 | Baseline interprétable |
| Random Forest | ≥ 0,75 | ≥ 0,75 | Modèle non-linéaire de référence |
| **XGBoost** | **≥ 0,82** | **≥ 0,85** | **Champion supervisé visé** |
| AutoEncoder | ≥ 0,30 | ≥ 0,30 | Sans étiquettes — objectif plus modeste |

### Résultats obtenus

Les résultats ci-dessous sont issus de l'ensemble de test (30 000 transactions, 39 fraudes) :

| Modèle | Recall | Precision | F1 | PR-AUC |
|---|---|---|---|---|
| isFlaggedFraud (baseline) | 0,004 | 1,000 | 0,008 | — |
| LR_balanced | 0,641 | 0,658 | 0,649 | 0,731 |
| LR_smote | 0,692 | 0,730 | 0,711 | 0,770 |
| RF_balanced | 0,769 | 0,833 | 0,800 | 0,849 |
| RF_smote | 0,795 | 0,816 | 0,805 | 0,859 |
| **XGB_smote** | **0,846** | **0,825** | **0,835** | **0,868** |
| Isolation Forest | 0,333 | 0,361 | 0,347 | 0,273 |
| AutoEncoder | 0,359 | 0,583 | 0,452 | 0,451 |

> **Tous les seuils cibles sont atteints.** XGBoost avec SMOTE est le modèle champion.

## 1.7 Contraintes et hypothèses

### Contraintes techniques

| Contrainte | Description | Impact sur le projet |
|---|---|---|
| **Volume de données** | 6,36M transactions — ne tient pas en mémoire vive | Échantillonnage stratifié à 200k tx |
| **Déséquilibre extrême** | 0,13 % de fraudes (ratio 1:774) | SMOTE + class_weight obligatoires |
| **Absence de GPU garantie** | Environnement de développement CPU-only possible | PyTorch avec fallback CPU automatique |
| **API LLM gratuite** | Groq free tier : 14 400 req/jour, 6 000 tokens/min | Limite à 20 explications LLM par run |
| **Reproductibilité** | Le pipeline doit être re-exécutable de bout en bout | `random_state=42` partout, seeds fixées |
| **Délai de projet** | Stage de 6 mois | Scope limité à 7 notebooks + pipeline src/ |

### Contraintes réglementaires et éthiques

| Contrainte | Description |
|---|---|
| **Confidentialité des données** | Les données client sont anonymisées conformément aux accords de confidentialité PwC |
| **Explicabilité obligatoire** | Toute décision de blocage doit être justifiable (RGPD Article 22) |
| **Pas de discrimination** | Les features utilisées ne doivent pas créer de biais protégés |
| **Audit trail** | Chaque prédiction doit être loggée avec son score et ses features |

### Hypothèses de travail

| # | Hypothèse | Vérifiable |
|---|---|---|
| H1 | Les étiquettes `isFraud` fournies par le client sont fiables | Oui (validées par les équipes métier du client) |
| H2 | La distribution des 200k tx échantillonnées est représentative des 6,36M | Oui (échantillonnage stratifié) |
| H3 | Les patterns de fraude sont stables dans le temps (pas de concept drift sur la période couverte) | Hypothèse simplificatrice |
| H4 | La variable `isFlaggedFraud` représente le système de détection actuel du client | Oui (confirmé par le client) |
| H5 | Un seuil d'alerte fixe est suffisant en production initiale | Oui (ajustable post-déploiement) |

## 1.8 Risques du projet

### Matrice des risques

| # | Risque | Probabilité | Impact | Criticité | Mitigation |
|---|---|---|---|---|---|
| R1 | **Surapprentissage** sur les données d'entraînement | Moyenne | Élevé | **Haute** | Validation holdout 15%, early stopping, régularisation L2 |
| R2 | **Data leakage** (fuites d'information temporelles) | Haute | Élevé | **Critique** | Suppression de `nameOrig`, `nameDest`, `isFlaggedFraud` ; scaler fit sur train only |
| R3 | **Concept drift** en production (les fraudeurs s'adaptent) | Haute | Élevé | **Critique** | Monitoring des scores, ré-entraînement périodique prévu |
| R4 | **Déséquilibre non géré** → modèle biaisé vers la classe majoritaire | Haute | Élevé | **Critique** | SMOTE + class_weight + F2-score comme critère d'optimisation |
| R5 | **Seuil mal calibré** → trop de faux positifs ou faux négatifs | Moyenne | Moyen | **Moyenne** | Optimisation du seuil sur validation set (maximisation F2) |
| R6 | **Disponibilité de l'API Groq** en production | Faible | Moyen | **Faible** | Fallback : explications SHAP/LIME sans LLM |
| R7 | **Coût computationnel** de l'AutoEncoder en production | Faible | Faible | **Faible** | Batch inference ; scoring < 10ms par transaction en CPU |
| R8 | **Non-interprétabilité** de XGBoost pour les décideurs | Haute | Moyen | **Moyenne** | SHAP global + LIME local + explications LLM |

### Focus sur les risques critiques

#### R2 — Data Leakage

Le risque de fuite d'information est le plus insidieux dans ce projet. Les variables suivantes ont été **explicitement supprimées** :
- `nameOrig`, `nameDest` : identifiants uniques → mémorisation des entités plutôt qu'apprentissage de patterns
- `isFlaggedFraud` : variable target dérivée, crée une fuite directe
- Balances brutes (`oldbalanceOrg`, `newbalanceOrig`, etc.) : remplacées par des différences et ratios pour éviter la mémorisation
- `step` brut : remplacé par `hour` et `day` extraits

#### R3 — Concept Drift

Les données fournies par le client couvrent 30 jours consécutifs d'activité. Le modèle est entraîné sur les 70 premiers pourcents de la période, évalué sur les 30 % restants. En production réelle :
- Les fraudeurs adaptent leurs stratégies aux contre-mesures déployées
- Un système de **monitoring continu** des distributions de scores est recommandé
- Un **ré-entraînement mensuel** avec les nouvelles fraudes confirmées est préconisé

### Plan de gestion des risques

```
Phase 1 (NB03–NB06) : Mitigation technique
  ├── Validation stricte sur holdout
  ├── Suppression systématique des features à risque de leakage
  └── Optimisation du seuil sur validation, évaluation finale sur test

Phase 2 (Post-déploiement) : Monitoring
  ├── Suivi hebdomadaire du score de fraude moyen
  ├── Alerte si distribution des features dérive > 2σ
  └── Revue mensuelle des faux négatifs confirmés
```

---

## Résumé — Business Understanding

| Dimension | Synthèse |
|---|---|
| **Problème** | Détection de fraudes dans 200k transactions de paiement mobile (ratio 1:774) |
| **Approche** | Double stratégie : supervisée (XGBoost) + non supervisée (AutoEncoder) |
| **Mesure de succès** | Recall ≥ 80 % sur le test set avec PR-AUC ≥ 0,85 |
| **Contrainte majeure** | Déséquilibre extrême + risque de data leakage |
| **Risque principal** | Concept drift en production → monitoring continu requis |
| **Valeur ajoutée** | Explicabilité (SHAP/LIME) + rapports LLM automatiques |

> **Prochaine étape :** [Notebook 02 — Data Understanding](02_data_understanding.ipynb) — Exploration et analyse des données brutes.